In [1]:
# import current working diretories

import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research


In [2]:
# system path

sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction")

In [3]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction\\research'

In [4]:
# changing to the parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction'

In [6]:
# import box versions

import box
print(box.__version__)

7.4.1


In [7]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PredictionPipelineConfig:

    model_path: Path
    preprocessor_unscaled_path: Path

In [8]:
from Hotel_Booking_Cancellation_Prediction.constant import *
from Hotel_Booking_Cancellation_Prediction.utils.common import read_yaml, create_directories
from Hotel_Booking_Cancellation_Prediction.entity.config_entity import PredictionPipelineConfig
from Hotel_Booking_Cancellation_Prediction.config.configuration import ConfigurationManager

In [9]:
# configuration manager

def get_prediction_pipeline_config(self) -> PredictionPipelineConfig:

    config = self.config["prediction_pipeline"]

    prediction_pipeline_config = PredictionPipelineConfig(

        model_path=Path(config.model_path),

        preprocessor_unscaled_path =Path(config.preprocessor_unscaled_path)
    )

    return prediction_pipeline_config

In [10]:
# components

import pandas as pd
import joblib


class PredictionPipeline:

    def __init__(self, config):

        self.config = config

    # LOAD MODEL

    def load_model(self):

        model = joblib.load(self.config.model_path)

        return model

    # LOAD PREPROCESSOR

    def load_preprocessor(self):

        preprocessor = joblib.load(
            self.config.preprocessor_unscaled_path
        )

        return preprocessor

    # MAKE PREDICTION

    def predict(self, input_data):

        # Load trained model

        model = self.load_model()

        # Load saved preprocessor

        preprocessor = self.load_preprocessor()

        # Convert user input into DataFrame

        input_df = pd.DataFrame([input_data])

        # Apply the SAME preprocessing used during model training i.e., unscaled data

        transformed_input = (
            preprocessor.transform(
                input_df
            )
        )

        # Make prediction

        prediction = model.predict(
            transformed_input)[0]

        # Get prediction probability

        if hasattr(
            model,
            "predict_proba"):

            probability = model.predict_proba(
                transformed_input
            )[0][1]

        else:

            probability = None

        return prediction, probability

In [11]:
from Hotel_Booking_Cancellation_Prediction.logging import logger

In [12]:
# pipeline

class PredictionPipelineTrainingPipeline:

    def __init__(self):
        pass

    def main(self,input_data):

        try:

            logger.info(">>>>>> Prediction Pipeline Stage Started <<<<<<")

            config = ConfigurationManager()

            prediction_pipeline_config = (config.get_prediction_pipeline_config() )

            prediction_pipeline = PredictionPipeline(config=prediction_pipeline_config)

            prediction, probability = (prediction_pipeline.predict(input_data))

            logger.info(">>>>>> Prediction Pipeline Stage Completed <<<<<<")

            return prediction, probability


        except Exception as e:

            logger.exception(e)
            raise e

In [13]:
import pandas as pd

df = pd.read_csv("artifacts/data_ingestion/hotel_bookings.csv")

input_data = df.drop(columns=["is_canceled"]).iloc[0].to_dict()

# Create the same engineered features used during training

input_data["total_nights"] = (
    input_data["stays_in_weekend_nights"]
    + input_data["stays_in_week_nights"]
)

input_data["total_guests"] = (
    input_data["adults"]
    + input_data["children"]
    + input_data["babies"]
)

print(input_data)

# these columns are missing because in data transformation we have changed the 2 columns to group as 1
#  so we need the old raw data for prediction input so we created again

{'hotel': 'Resort Hotel', 'lead_time': 342, 'arrival_date_year': 2015, 'arrival_date_month': 'July', 'arrival_date_week_number': 27, 'arrival_date_day_of_month': 1, 'stays_in_weekend_nights': 0, 'stays_in_week_nights': 0, 'adults': 2, 'children': 0.0, 'babies': 0, 'meal': 'BB', 'country': 'PRT', 'market_segment': 'Direct', 'distribution_channel': 'Direct', 'is_repeated_guest': 0, 'previous_cancellations': 0, 'previous_bookings_not_canceled': 0, 'reserved_room_type': 'C', 'assigned_room_type': 'C', 'booking_changes': 3, 'deposit_type': 'No Deposit', 'agent': nan, 'company': nan, 'days_in_waiting_list': 0, 'customer_type': 'Transient', 'adr': 0.0, 'required_car_parking_spaces': 0, 'total_of_special_requests': 0, 'reservation_status': 'Check-Out', 'reservation_status_date': '2015-07-01', 'total_nights': 0, 'total_guests': 2.0}


In [14]:
obj = PredictionPipelineTrainingPipeline()

prediction, probability = obj.main(input_data)

print("Prediction:", prediction)
print("Probability:", probability)

[2026-08-19 00:27:10,629: INFO: 930679976: >>>>>> Prediction Pipeline Stage Started <<<<<<]
[2026-08-19 00:27:10,645: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\config\config.yaml loaded successfully]
[2026-08-19 00:27:10,649: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\params.yaml loaded successfully]
[2026-08-19 00:27:10,651: INFO: common: created directory at artifacts]


c:\Users\Greesha Vaishnavi\.conda\envs\class\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Greesha Vaishnavi\.conda\envs\class\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


[2026-08-19 00:27:11,922: INFO: 930679976: >>>>>> Prediction Pipeline Stage Completed <<<<<<]
Prediction: 0
Probability: 0.055


prediction = 0 means This booking will NOT be cancelled

probabilty = 0.055 means model estimates 5.5% probability of cancellation